# Earthapps Jezero research workflow

This CPU-compatible workflow reads the existing Earthapps checkout, validates its manifest and binary assets, reproduces current calculations, and compares research alternatives. It never downloads scientific data or writes production assets.

## Private Colab setup

Upload a checkout or mount a private Drive folder. Do not paste GitHub credentials or tokens into this notebook.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path('/content/Earthapps')  # change to the uploaded or Drive-mounted checkout
DATA_ROOT = REPO_ROOT / 'public' / 'data' / 'jezero'
OUTPUT_ROOT = REPO_ROOT / 'analysis' / 'outputs'
sys.path.insert(0, str(REPO_ROOT))
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
# Run only when package installation has been approved.
# %pip install -r /content/Earthapps/analysis/requirements-colab.txt

In [ ]:
import json
import time
import numpy as np
import matplotlib.pyplot as plt
from analysis.io import load_jezero
from analysis.terrain_metrics import quality_report, local_relief
from analysis.slope_experiments import slope_experiment
from analysis.hazard_experiments import hazard_experiment
from analysis.reporting import write_report, save_map
from analysis.provenance import report_metadata

dataset = load_jezero(DATA_ROOT, verify_hashes=True)
print(dataset.manifest['datasetId'])
print(dataset.analysis_grid)
print(dataset.manifest['sourceCrs'])
print(dataset.manifest.get('verticalDatum'))

## 1. Dataset integrity and spatial quality

In [ ]:
quality = quality_report(dataset)
write_report(OUTPUT_ROOT / 'quality-report.json', quality)
quality

In [ ]:
extent = dataset.analysis_grid.extent
save_map(OUTPUT_ROOT / 'height-map.png', dataset.heights_m, 'HiRISE analysis elevation (m)', extent=extent)
save_map(OUTPUT_ROOT / 'slope-baseline-map.png', dataset.slope_deg, 'Bundled slope (degrees)', 'magma', extent=extent)
save_map(OUTPUT_ROOT / 'thermal-map.png', np.where(dataset.thermal_valid, dataset.thermal_dn, np.nan), 'THEMIS relative DN', 'inferno', extent=extent)

## 2–3. Baseline slope reproduction and alternative estimator

In [ ]:
slopes = slope_experiment(dataset)
write_report(OUTPUT_ROOT / 'slope-comparison.json', slopes)
for name in ('baselineDeg', 'hornDeg'):
    save_map(OUTPUT_ROOT / f'{name}.png', slopes[name], name, 'magma', extent=extent)
slopes['baseline'], slopes['horn']

Horn slope is a sensitivity experiment. Without independent slope ground truth, lower visual noise is not evidence of greater accuracy.

## 4. Hazard classification comparisons

In [ ]:
hazards = hazard_experiment(slopes['baselineDeg'])
write_report(OUTPUT_ROOT / 'hazard-comparison.json', hazards)
hazards

The comparison explicitly exposes the existing 15° threshold and four-neighbor behavior versus eight-neighbor connectivity. These are illustrative caution zones, not confirmed hazards.

## 5. Route-cost experiments

Populate this cell from an approved route export or the mission JSON. The coefficients remain scenario assumptions because no validated astronaut mobility model is bundled.

In [ ]:
mission = json.loads((DATA_ROOT / 'mission.json').read_text(encoding='utf-8'))
from analysis.route_costs import route_experiment
from analysis.route_terrain import analyze_route, load_route_points
approved_export = REPO_ROOT / 'analysis' / 'inputs' / 'route-export.json'
if approved_export.exists():
    points = load_route_points(approved_export)
    route_label = 'approved route export'
else:
    # presetNodes are mission points of interest in application order, not an approved
    # surveyed route export. This result is explicitly illustrative.
    points = [(n['point']['x'], n['point']['y']) for n in mission['presetNodes']]
    route_label = 'illustrative mission presetNodes'
terrain_route = analyze_route(dataset, points, route_label=route_label)
write_report(OUTPUT_ROOT / 'terrain-route-report.json', terrain_route)
segment_slopes = [float(np.nanmean(slopes['baselineDeg']))] * (len(points) - 1)
route_report = route_experiment(points, segment_slopes, speed_mps=0.5, stop_count=3, dwell_min=10)
write_report(OUTPUT_ROOT / 'route-cost-comparison.json', route_report)
terrain_route['routeLabel'], terrain_route['totals'], route_report

## 6. Cross-layer registration checks

In [ ]:
from analysis.registration import registration_report
registration = registration_report(dataset)
write_report(OUTPUT_ROOT / 'registration-report.json', registration)
registration

## 7. Reproducibility and performance

In [ ]:
start = time.perf_counter()
_ = slope_experiment(dataset)
elapsed = time.perf_counter() - start
provenance = report_metadata(dataset)
provenance['slopeExperimentWallTimeSec'] = elapsed
provenance['memoryBytes'] = {
    'heights': int(dataset.heights_m.nbytes),
    'relief': int(dataset.relief_m.nbytes) if dataset.relief_m is not None else 0,
    'slope': int(dataset.slope_encoded.nbytes),
    'thermal': int(dataset.thermal_dn.nbytes),
}
write_report(OUTPUT_ROOT / 'provenance.json', provenance)
provenance

## Interpretation and limitations

HiRISE elevations are source-derived meters above the Mars 2000 equipotential datum. THEMIS values are relative daytime IR brightness, not temperature. The route model is illustrative, uses horizontal distance and configured speed, and does not model astronaut physiology, obstacles, communications, weather, radiation, or battery/oxygen constraints. Alternative methods are integration candidates only if their assumptions and quantitative effects are accepted.